# ECLIPSE Stronger Frozen Backbone - ConvNeXt-V2-Tiny Colab Version

This is the cleaned no-TTA ConvNeXt-V2-Tiny notebook. Run it from **Cell 0 to the final cell in order** in VS Code while connected to a Google Colab GPU runtime.

Ordered flow:

1. Mount Drive and clone/reuse ECLIPSE.
2. Install the pinned Torch/CUDA stack and dependencies.
3. Build Detectron2 from source with hardened CUDA checks.
4. Restore/download/prepare ADE20K, caching it in Drive.
5. Install/check ConvNeXt-V2-Tiny from timm.
6. Patch ECLIPSE with the ConvNeXt-V2-Tiny backbone and generate no-TTA scripts.
7. Build ECLIPSE custom CUDA ops.
8. Run one sanity check.
9. Run the clearly labeled restore/continue checkpoint cell before training.
10. Train/resume base, then 100-50, 100-10, and 100-5.
11. Extract PQ and graph article results vs your ConvNeXt-V2-Tiny run.

Checkpoint behavior:

- Base training writes checkpoints every 10k iterations and live-syncs them to Drive while training runs.
- 100-50 training also writes checkpoints every 10k iterations and live-syncs them to Drive while training runs.
- The restore/continue cell restores final checkpoints plus live 10k checkpoints and `last_checkpoint`, so rerunning can continue from where Colab stopped instead of starting over.
- Drive checkpoint folder: `/content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_CHECKPOINTS/convnextv2_t`.


In [ ]:
# Cell 0 - Mount Drive and clone/reuse ECLIPSE

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

REPO_DIR = Path('/content/ECLIPSE_CONVNEXTV2_TINY')

%cd /content

# Keep the repo if it already exists in this runtime so reruns do not wipe local
# checkpoints/logs. Fresh Colab runtimes will clone it automatically.
if not (REPO_DIR / 'train_inc.py').exists():
    !git clone https://github.com/clovaai/ECLIPSE.git ECLIPSE_CONVNEXTV2_TINY
else:
    print('Reusing existing ECLIPSE checkout:', REPO_DIR)

%cd /content/ECLIPSE_CONVNEXTV2_TINY
!pwd
!ls


In [ ]:

# Cell 1 - Install a compatible Torch/CUDA stack

%cd /content/ECLIPSE_CONVNEXTV2_TINY

# Current Colab system CUDA is usually 12.8.
# We pin PyTorch to CUDA 12.8, not CUDA 13.0, to avoid Detectron2 compile mismatch.
!python -m pip uninstall -y torch torchvision torchaudio detectron2

!python -m pip install -U pip setuptools wheel ninja cython

!python -m pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128

!nvcc --version

import torch

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert torch.version.cuda == "12.8", "Torch CUDA is not 12.8. Stop here."


In [ ]:
# Cell 2 - Install Python dependencies without upgrading Torch

%cd /content/ECLIPSE_CONVNEXTV2_TINY

!python -m pip install -U pip setuptools wheel ninja cython
!python -m pip install "numpy<2" "cython<3" pycocotools

# Install common Detectron2/ECLIPSE dependencies.
!python -m pip install \
    fvcore iopath omegaconf hydra-core cloudpickle termcolor yacs tabulate tqdm tensorboard \
    scipy scikit-image matplotlib pillow opencv-python-headless einops timm regex ftfy \
    git+https://github.com/cocodataset/panopticapi.git \
    git+https://github.com/mcordts/cityscapesScripts.git

# If ECLIPSE has requirements.txt, install it but ignore packages that can break the environment.
from pathlib import Path

req = Path("requirements.txt")

if req.exists():
    banned = ("torch", "torchvision", "torchaudio", "detectron2", "numpy", "opencv", "mmcv")
    clean_lines = []

    for line in req.read_text().splitlines():
        s = line.strip()

        if not s or s.startswith("#"):
            continue

        if s.lower().startswith(banned):
            print("Skipping requirement:", s)
            continue

        clean_lines.append(s)

    Path("/content/eclipse_requirements_clean.txt").write_text("\n".join(clean_lines))

    print("Installing cleaned requirements:")
    print(Path("/content/eclipse_requirements_clean.txt").read_text())

    !python -m pip install -r /content/eclipse_requirements_clean.txt

# Verify that Torch did not get upgraded.
import torch

print("Torch after dependencies:", torch.__version__)
print("Torch CUDA after dependencies:", torch.version.cuda)

assert torch.version.cuda == "12.8", "Torch changed. Stop and rerun from Cell 1."


In [ ]:
# Cell 3 - Build and verify Detectron2 from source, hardened for Colab CUDA

%cd /content

import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from torch.utils.cpp_extension import CUDA_HOME

# Detectron2 trouble usually comes from one of three things:
# 1. a stale compiled .so from a previous Torch/CUDA version;
# 2. pip importing a different detectron2 package than the source tree you built;
# 3. CUDA_HOME/nvcc not matching the Torch CUDA runtime closely enough.
# This cell avoids those by doing a clean source checkout/build, installing with
# --no-deps so Torch is not changed, and forcing /content/detectron2_src to the
# front of sys.path before importing detectron2._C.

DETECTRON2_SRC = Path("/content/detectron2_src")
ECLIPSE_SRC = Path("/content/ECLIPSE_CONVNEXTV2_TINY")
DETECTRON2_REPO = "https://github.com/facebookresearch/detectron2.git"
DETECTRON2_REF = "main"  # Change only if you intentionally want to pin a known commit.


def run(cmd, cwd=None, env=None):
    cwd = Path(cwd or "/content")
    print(f"\n$ {cmd}\n[cwd] {cwd}")
    subprocess.run(["bash", "-lc", cmd], cwd=str(cwd), env=env, check=True)


def nvcc_version_text():
    if shutil.which("nvcc") is None:
        return "nvcc not found"
    try:
        return subprocess.check_output(["nvcc", "--version"], text=True, stderr=subprocess.STDOUT)
    except Exception as exc:
        return f"nvcc check failed: {exc}"


def cuda_version_tuple(text):
    match = re.search(r"release\s+(\d+)\.(\d+)", text)
    if not match:
        return None
    return int(match.group(1)), int(match.group(2))

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA_HOME:", CUDA_HOME)
print("nvcc:\n", nvcc_version_text())

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot see a CUDA GPU. Stop here before building Detectron2.")
if CUDA_HOME is None:
    raise RuntimeError("CUDA_HOME is None. Colab CUDA toolkit is not visible; Detectron2 cannot build its CUDA extension.")

nvcc_tuple = cuda_version_tuple(nvcc_version_text())
torch_tuple = tuple(map(int, torch.version.cuda.split(".")[:2])) if torch.version.cuda else None
if nvcc_tuple and torch_tuple and nvcc_tuple[0] != torch_tuple[0]:
    raise RuntimeError(f"CUDA major mismatch: nvcc={nvcc_tuple}, torch CUDA={torch_tuple}. Reinstall matching Torch/CUDA first.")
if nvcc_tuple and torch_tuple and nvcc_tuple != torch_tuple:
    print(f"WARNING: nvcc minor version {nvcc_tuple} differs from Torch CUDA {torch_tuple}. Same major is usually OK, but if build fails, restart runtime and rerun Cell 1.")

major, minor = torch.cuda.get_device_capability(0)
torch_cuda_arch_list = f"{major}.{minor}"
print("GPU:", torch.cuda.get_device_name(0))
print("TORCH_CUDA_ARCH_LIST:", torch_cuda_arch_list)

# Remove any pip-installed detectron2 and any stale source checkout. This prevents
# the notebook from accidentally importing an old compiled extension.
run(f"{sys.executable} -m pip uninstall -y detectron2")
run("rm -rf /content/detectron2_src")
run(f"git clone --depth 1 --branch {DETECTRON2_REF} {DETECTRON2_REPO} {DETECTRON2_SRC}")
run("git rev-parse --short HEAD", cwd=DETECTRON2_SRC)

# Clean all compiled artifacts before building against the current Torch/CUDA.
run("rm -rf build && find . -name '*.so' -delete && find . -name '*.pyd' -delete", cwd=DETECTRON2_SRC)

build_env = os.environ.copy()
build_env["FORCE_CUDA"] = "1"
build_env["TORCH_CUDA_ARCH_LIST"] = torch_cuda_arch_list

try:
    run(f"{sys.executable} setup.py build_ext --inplace", cwd=DETECTRON2_SRC, env=build_env)
    run(f"{sys.executable} -m pip install -e . --no-build-isolation --no-deps", cwd=DETECTRON2_SRC, env=build_env)
except subprocess.CalledProcessError:
    print("\nDetectron2 build failed. Restart the runtime, rerun Cells 1-3, and do not continue until this cell prints Detectron2 _C OK.")
    raise

# Force this notebook kernel to import the freshly built source tree.
for module_name in list(sys.modules):
    if module_name == "detectron2" or module_name.startswith("detectron2."):
        del sys.modules[module_name]

for path in [str(DETECTRON2_SRC), str(ECLIPSE_SRC)]:
    if path not in sys.path:
        sys.path.insert(0, path)
os.environ["PYTHONPATH"] = f"{DETECTRON2_SRC}:{ECLIPSE_SRC}:" + os.environ.get("PYTHONPATH", "")

import detectron2
print("Detectron2 imported from:", detectron2.__file__)

from detectron2 import _C
print("Detectron2 _C OK. Source build is usable.")


In [ ]:
# Cell 4 - Force notebook and shell to use the freshly built Detectron2

%cd /content/ECLIPSE_CONVNEXTV2_TINY

import os
import sys
from pathlib import Path

# This cell prevents a subtle but common problem: Python importing a stale pip
# detectron2 instead of the freshly compiled /content/detectron2_src checkout.
# It clears loaded detectron2 modules, puts the source checkout first in sys.path,
# sets PYTHONPATH for shell training commands, and verifies detectron2._C again.

DETECTRON2_SRC = Path("/content/detectron2_src")
ECLIPSE_SRC = Path("/content/ECLIPSE_CONVNEXTV2_TINY")

if not (DETECTRON2_SRC / "detectron2").exists():
    raise FileNotFoundError("/content/detectron2_src is missing. Rerun Cell 3.")

for module_name in list(sys.modules):
    if module_name == "detectron2" or module_name.startswith("detectron2."):
        del sys.modules[module_name]

paths_to_add = [str(DETECTRON2_SRC), str(ECLIPSE_SRC)]
for path in reversed(paths_to_add):
    if path in sys.path:
        sys.path.remove(path)
for path in reversed(paths_to_add):
    sys.path.insert(0, path)

os.environ["PYTHONPATH"] = ":".join(paths_to_add + [os.environ.get("PYTHONPATH", "")])
%env PYTHONPATH=/content/detectron2_src:/content/ECLIPSE_CONVNEXTV2_TINY

import detectron2
print("Detectron2:", detectron2.__file__)
if not str(detectron2.__file__).startswith(str(DETECTRON2_SRC)):
    raise RuntimeError(f"Wrong Detectron2 import path: {detectron2.__file__}")

from detectron2 import _C
print("Detectron2 _C OK")
print("Detectron2 is locked to the freshly built source checkout for the notebook and shell cells.")


In [ ]:
# Cell 5 - Restore/download/prepare ADE20K with progress and ETA

%cd /content/ECLIPSE_CONVNEXTV2_TINY

from pathlib import Path
from shlex import quote as shlex_quote
import shutil
import subprocess
import time

# Why this cell is written this way:
# Google Drive is very slow when copying thousands of small ADE20K files with
# shutil.copytree(). It can sit for an hour with no output. This cell gives you
# progress/ETA and creates a single tar archive cache so future restores are much
# faster and easier to monitor.
#
# Restore priority:
# 1. If a prepared tar archive exists in Drive, stream it through pv with ETA and
#    extract it into /content.
# 2. If only the old Drive folder cache exists, restore it with rsync progress.
#    After that, create the tar archive cache for future runs.
# 3. If no cache exists, download ADE20K, prepare it, then create the tar cache.

DATASETS_ROOT = Path('/content/ECLIPSE_CONVNEXTV2_TINY/datasets')
DATASET_DIR = DATASETS_ROOT / 'ADEChallengeData2016'
DRIVE_CACHE_ROOT = Path('/content/drive/MyDrive/ECLIPSE_CACHE')
DRIVE_DATASET_DIR = DRIVE_CACHE_ROOT / 'ADEChallengeData2016'
DRIVE_TAR = DRIVE_CACHE_ROOT / 'ADEChallengeData2016_prepared.tar'
DRIVE_TAR_TMP = DRIVE_CACHE_ROOT / 'ADEChallengeData2016_prepared.tar.tmp'
PREP_MARKER = DATASET_DIR / '.eclipse_prepare_done'

DATASETS_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def q(path):
    return shlex_quote(str(path))


def hms(seconds):
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, seconds = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes}m {seconds}s"
    if minutes:
        return f"{minutes}m {seconds}s"
    return f"{seconds}s"


def run(cmd, cwd=None):
    # subprocess output is intentionally not captured so rsync/pv progress and ETA
    # stream directly into the notebook output while the command is still running.
    cwd = Path(cwd or '/content/ECLIPSE_CONVNEXTV2_TINY')
    print(f"\n$ {cmd}\n[cwd] {cwd}")
    started = time.time()
    subprocess.run(['bash', '-lc', cmd], cwd=str(cwd), check=True)
    print(f"Finished in {hms(time.time() - started)}")


def ensure_progress_tools():
    # rsync provides progress for folder restore; pv provides progress, speed, and
    # ETA for tar archive restore/create. Install only if Colab does not have them.
    missing = []
    if shutil.which('rsync') is None:
        missing.append('rsync')
    if shutil.which('pv') is None:
        missing.append('pv')
    if missing:
        print('Installing progress tools:', ', '.join(missing))
        run('apt-get -qq update && apt-get -qq install -y ' + ' '.join(missing), cwd='/content')


def raw_ade20k_exists():
    required = [
        DATASET_DIR / 'images' / 'training',
        DATASET_DIR / 'images' / 'validation',
        DATASET_DIR / 'annotations' / 'training',
        DATASET_DIR / 'annotations' / 'validation',
        DATASET_DIR / 'annotations_instance',
    ]
    return all(path.exists() for path in required)


def remove_incomplete_local_dataset_if_needed():
    # If a previous silent copy was interrupted, the folder may exist but be
    # incomplete. Remove that partial /content copy so restore/download can start
    # from a known-good state. This does not delete anything from Google Drive.
    if DATASET_DIR.exists() and not raw_ade20k_exists():
        print('Found an incomplete local ADE20K folder from an interrupted restore.')
        print('Removing incomplete /content copy only:', DATASET_DIR)
        shutil.rmtree(DATASET_DIR)


def restore_from_drive_tar():
    print('Restoring ADE20K from Drive tar archive with progress/ETA:')
    print(' ', DRIVE_TAR)
    run(f'mkdir -p {q(DATASETS_ROOT)} && pv -ptebar {q(DRIVE_TAR)} | tar -xf - -C {q(DATASETS_ROOT)}')


def restore_from_drive_folder_with_progress():
    print('Restoring ADE20K from old Drive folder cache with rsync progress.')
    print('This is still slower than the tar archive, but now you will see progress.')
    src = str(DRIVE_DATASET_DIR) + '/'
    dst = str(DATASET_DIR) + '/'
    run(f'mkdir -p {q(DATASET_DIR)} && rsync -a --human-readable --info=progress2 {q(src)} {q(dst)}')


def download_raw_ade20k():
    print('No usable Drive ADE20K cache found. Downloading raw ADE20K files with progress.')
    run('wget -c --show-progress -O ADEChallengeData2016.zip http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip', cwd=DATASETS_ROOT)
    run('unzip -q ADEChallengeData2016.zip && rm ADEChallengeData2016.zip', cwd=DATASETS_ROOT)
    run('wget -c --show-progress -O annotations_instance.tar http://sceneparsing.csail.mit.edu/data/ChallengeData2017/annotations_instance.tar', cwd=DATASET_DIR)
    run('tar -xf annotations_instance.tar && rm annotations_instance.tar', cwd=DATASET_DIR)


def prepare_ade20k_if_needed():
    if PREP_MARKER.exists():
        print('ADE20K prepare marker exists, skipping prepare scripts:', PREP_MARKER)
        return
    print('Running ECLIPSE ADE20K preparation scripts...')
    run('python datasets/prepare_ade20k_sem_seg.py')
    run('python datasets/prepare_ade20k_pan_seg.py')
    run('python datasets/prepare_ade20k_ins_seg.py')
    PREP_MARKER.write_text('prepared\n')
    print('Wrote prepare marker:', PREP_MARKER)


def create_drive_tar_cache_if_needed():
    if DRIVE_TAR.exists() and DRIVE_TAR.stat().st_size > 1_000_000:
        size_gb = DRIVE_TAR.stat().st_size / 1024 / 1024 / 1024
        print(f'Drive tar cache already exists: {DRIVE_TAR} ({size_gb:.2f} GB)')
        return
    print('Creating single-file Drive tar cache for future fast restores.')
    print('This replaces slow future folder copytree restores with a progress/ETA tar restore.')
    size_bytes = subprocess.check_output(['du', '-sb', str(DATASET_DIR)], text=True).split()[0]
    if DRIVE_TAR_TMP.exists():
        DRIVE_TAR_TMP.unlink()
    cmd = f'tar -cf - -C {q(DATASETS_ROOT)} ADEChallengeData2016 | pv -ptebar -s {size_bytes} > {q(DRIVE_TAR_TMP)} && mv {q(DRIVE_TAR_TMP)} {q(DRIVE_TAR)}'
    run(cmd)
    size_gb = DRIVE_TAR.stat().st_size / 1024 / 1024 / 1024
    print(f'Created Drive tar cache: {DRIVE_TAR} ({size_gb:.2f} GB)')


ensure_progress_tools()
remove_incomplete_local_dataset_if_needed()

if raw_ade20k_exists():
    print('Raw ADE20K already exists locally in /content; skipping restore/download.')
elif DRIVE_TAR.exists() and DRIVE_TAR.stat().st_size > 1_000_000:
    restore_from_drive_tar()
elif DRIVE_DATASET_DIR.exists():
    restore_from_drive_folder_with_progress()
else:
    download_raw_ade20k()

if not raw_ade20k_exists():
    raise FileNotFoundError('ADE20K restore/download finished, but required raw folders are still missing. Check the progress output above.')

prepare_ade20k_if_needed()
create_drive_tar_cache_if_needed()

print('ADE20K preparation finished.')
print('Future runs should restore from this progress/ETA archive:', DRIVE_TAR)


In [ ]:
# Cell 6 - Install/check ConvNeXt-V2-Tiny from timm

%cd /content/ECLIPSE_CONVNEXTV2_TINY

# timm hosts ConvNeXt-V2 model definitions and pretrained weights.
# This should not upgrade Torch; it only makes sure the model catalog is recent enough.
!python -m pip install -U "timm>=1.0.15" "huggingface_hub>=0.23"

import timm
import torch

BACKBONE_TAG = "convnextv2_t"
BACKBONE_LABEL = "ConvNeXt-V2-Tiny"

preferred_models = [
    "convnextv2_tiny.fcmae_ft_in22k_in1k",
    "convnextv2_tiny.fcmae_ft_in1k",
    "convnextv2_tiny.fcmae",
]
available = set(timm.list_models("convnextv2_tiny*", pretrained=True))
print("Available pretrained ConvNeXt-V2-Tiny timm models:", sorted(available))

CONVNEXTV2_TIMM_MODEL = next((name for name in preferred_models if name in available), None)
if CONVNEXTV2_TIMM_MODEL is None:
    # Keep a deterministic fallback; timm will raise here if the local catalog lacks it.
    CONVNEXTV2_TIMM_MODEL = preferred_models[0]

print("Using timm model:", CONVNEXTV2_TIMM_MODEL)
probe = timm.create_model(
    CONVNEXTV2_TIMM_MODEL,
    pretrained=True,
    features_only=True,
    out_indices=(0, 1, 2, 3),
)
print("Feature channels:", probe.feature_info.channels())
print("Feature reductions:", probe.feature_info.reduction())
assert probe.feature_info.reduction() == [4, 8, 16, 32], "Expected FPN-like strides 4/8/16/32."
del probe
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Cell 7 - Patch ConvNeXt-V2 backbone and create no-TTA train/eval scripts

%cd /content/ECLIPSE_CONVNEXTV2_TINY

from pathlib import Path

try:
    BACKBONE_TAG
    BACKBONE_LABEL
    CONVNEXTV2_TIMM_MODEL
except NameError:
    BACKBONE_TAG = "convnextv2_t"
    BACKBONE_LABEL = "ConvNeXt-V2-Tiny"
    CONVNEXTV2_TIMM_MODEL = "convnextv2_tiny.fcmae_ft_in22k_in1k"

DATA_ROOT = "/content/ECLIPSE_CONVNEXTV2_TINY/datasets"
RESULT_ROOT = f"results/ade_ps_{BACKBONE_TAG}"
BASE_STEP_CKPT = f"results/ade_ps_{BACKBONE_TAG}_100_step0.pth"
LOG_DIR = "/content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_LOGS"
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

# Full-run defaults from the ECLIPSE scripts. Lower these only for a smoke test.
IMS_PER_BATCH = 8
BASE_MAX_ITER = 160000
# Base checkpoints are written every 10k iterations. Cell 10 live-syncs
# these intermediate checkpoints to Drive while the base stage is still training.
BASE_CHECKPOINT_PERIOD = 10000
# Avoid repeated validation during the long base stage; final eval still happens later.
BASE_EVAL_PERIOD = BASE_MAX_ITER
PROMPT_100_50_MAX_ITER = 80000
# 100-50 is also long enough to protect with intermediate checkpoints.
# Cell 10 live-syncs these to Drive while Cell 12 is training.
PROMPT_100_50_CHECKPOINT_PERIOD = 10000
PROMPT_100_10_MAX_ITER = 16000
PROMPT_100_5_MAX_ITER = 12000
NUM_QUERIES = 100

backbone_file = Path("mask2former/modeling/backbone/convnextv2.py")
backbone_file.write_text(f'''import os

import torch
from detectron2.modeling import BACKBONE_REGISTRY, Backbone, ShapeSpec
import timm


@BACKBONE_REGISTRY.register()
class D2ConvNeXtV2Tiny(Backbone):
    def __init__(self, cfg, input_shape):
        super().__init__()
        model_name = os.getenv("CONVNEXTV2_TIMM_MODEL", "{CONVNEXTV2_TIMM_MODEL}")
        self.body = timm.create_model(
            model_name,
            pretrained=True,
            features_only=True,
            out_indices=(0, 1, 2, 3),
        )
        self._out_features = ["res2", "res3", "res4", "res5"]
        channels = self.body.feature_info.channels()
        strides = self.body.feature_info.reduction()
        self._out_feature_channels = dict(zip(self._out_features, channels))
        self._out_feature_strides = dict(zip(self._out_features, strides))

    def forward(self, x):
        assert x.dim() == 4, f"ConvNeXt-V2 backbone expects NCHW input. Got {{x.shape}}."
        features = self.body(x)
        return {{name: feat for name, feat in zip(self._out_features, features)}}

    def output_shape(self):
        return {{
            name: ShapeSpec(
                channels=self._out_feature_channels[name],
                stride=self._out_feature_strides[name],
            )
            for name in self._out_features
        }}

    @property
    def size_divisibility(self):
        return 32
''')

modeling_init = Path("mask2former/modeling/__init__.py")
init_text = modeling_init.read_text()
import_line = "from .backbone.convnextv2 import D2ConvNeXtV2Tiny\n"
if import_line not in init_text:
    modeling_init.write_text(init_text + import_line)

!python -m py_compile mask2former/modeling/backbone/convnextv2.py
print("Patched ConvNeXt-V2 backbone:", backbone_file)

config_dir = Path("configs/ade20k/panoptic-segmentation/convnextv2")
config_dir.mkdir(parents=True, exist_ok=True)
CFG_FILE = str(config_dir / f"maskformer2_{BACKBONE_TAG}_bs16_160k.yaml")

convnext_config = f'''_BASE_: ../maskformer2_R50_bs16_160k.yaml
MODEL:
  BACKBONE:
    NAME: "D2ConvNeXtV2Tiny"
  WEIGHTS: ""
  PIXEL_MEAN: [123.675, 116.280, 103.530]
  PIXEL_STD: [58.395, 57.120, 57.375]
  MASK_FORMER:
    NUM_OBJECT_QUERIES: {NUM_QUERIES}
TEST:
  AUG:
    ENABLED: False
'''
Path(CFG_FILE).write_text(convnext_config)
print("Wrote config:", CFG_FILE)
print(Path(CFG_FILE).read_text())

script_dir = Path(f"script/ade_ps_{BACKBONE_TAG}")
script_dir.mkdir(parents=True, exist_ok=True)

common_header = r'''#!/bin/bash
set -e

export DETECTRON2_DATASETS="__DATA_ROOT__"
export CONVNEXTV2_TIMM_MODEL="__TIMM_MODEL__"
ngpus=$(nvidia-smi --list-gpus | wc -l)

cfg_file="__CFG_FILE__"
base="__RESULT_ROOT__"
base_step_ckpt="__BASE_STEP_CKPT__"

mkdir -p "${base}"
cont_args=""
dist_args=""

meth_args="MODEL.MASK_FORMER.TEST.MASK_BG False MODEL.MASK_FORMER.PER_PIXEL False MODEL.MASK_FORMER.FOCAL True TEST.AUG.ENABLED False SOLVER.IMS_PER_BATCH __IMS_PER_BATCH__"

base_queries=__NUM_QUERIES__
dice_weight=5.0
mask_weight=5.0
soft_mask=False
soft_cls=False
deep_cls=True
'''
common_header = (common_header
    .replace("__DATA_ROOT__", DATA_ROOT)
    .replace("__TIMM_MODEL__", CONVNEXTV2_TIMM_MODEL)
    .replace("__CFG_FILE__", CFG_FILE)
    .replace("__RESULT_ROOT__", RESULT_ROOT)
    .replace("__BASE_STEP_CKPT__", BASE_STEP_CKPT)
    .replace("__IMS_PER_BATCH__", str(IMS_PER_BATCH))
    .replace("__NUM_QUERIES__", str(NUM_QUERIES))
)

base_script = common_header + r'''
step_args="CONT.BASE_CLS 100 CONT.INC_CLS 50 CONT.MODE overlap SEED 42"
class_weight=2.0
base_lr=0.0001
iter=__BASE_MAX_ITER__
num_prompts=0
exp_name="adps___TAG___base100"

weight_args="MODEL.MASK_FORMER.NUM_OBJECT_QUERIES ${base_queries} MODEL.MASK_FORMER.DICE_WEIGHT ${dice_weight} MODEL.MASK_FORMER.MASK_WEIGHT ${mask_weight} MODEL.MASK_FORMER.CLASS_WEIGHT ${class_weight} MODEL.MASK_FORMER.SOFTMASK ${soft_mask} CONT.SOFTCLS ${soft_cls} CONT.NUM_PROMPTS ${num_prompts} CONT.DEEP_CLS ${deep_cls}"
comm_args="OUTPUT_DIR ${base} ${meth_args} ${step_args} ${weight_args}"
inc_args="CONT.TASK 0 SOLVER.BASE_LR ${base_lr} TEST.EVAL_PERIOD __BASE_EVAL_PERIOD__ SOLVER.CHECKPOINT_PERIOD __BASE_CHECKPOINT_PERIOD__ SOLVER.MAX_ITER ${iter}"

if [ -f "${base_step_ckpt}" ]; then
  echo "Base checkpoint already exists: ${base_step_ckpt}"
  exit 0
fi

python train_inc.py --resume --num-gpus ${ngpus} --config-file ${cfg_file} ${comm_args} ${inc_args} NAME ${exp_name} WANDB False

base_output="${base}/mya-pan_100-50-ov/${exp_name}/step0/model_final.pth"
if [ ! -f "${base_output}" ]; then
  echo "Expected base output not found: ${base_output}"
  exit 1
fi
cp "${base_output}" "${base_step_ckpt}"
echo "Saved shared 100-class base checkpoint to ${base_step_ckpt}"
'''
base_script = (base_script
    .replace("__BASE_MAX_ITER__", str(BASE_MAX_ITER))
    .replace("__BASE_EVAL_PERIOD__", str(BASE_EVAL_PERIOD))
    .replace("__BASE_CHECKPOINT_PERIOD__", str(BASE_CHECKPOINT_PERIOD))
    .replace("__TAG__", BACKBONE_TAG)
)


def scenario_script(suffix, inc_cls, final_task, prompt_count, train_iter, train_deltas, eval_deltas, eval_period, checkpoint_period=500000):
    task = f"mya-pan_100-{inc_cls}-ov"
    loop = ""
    if final_task > 1:
        loop_values = " ".join(str(t) for t in range(2, final_task + 1))
        loop = f'''
for t in {loop_values}; do
  inc_args="CONT.TASK ${{t}} SOLVER.MAX_ITER ${{iter}} SOLVER.BASE_LR ${{base_lr}} TEST.EVAL_PERIOD {eval_period} SOLVER.CHECKPOINT_PERIOD {checkpoint_period}"
  python train_inc.py --resume --num-gpus ${{ngpus}} --config-file ${{cfg_file}} ${{comm_args}} ${{inc_args}} ${{cont_args}} ${{dist_args}} ${{vpt_args}} NAME ${{exp_name}} WANDB False
done
'''
    return common_header + f'''
step_args="CONT.BASE_CLS 100 CONT.INC_CLS {inc_cls} CONT.MODE overlap SEED 42"
exp_name="adps_{BACKBONE_TAG}_{suffix}"
task="{task}"
final_ckpt="${{base}}/${{task}}/${{exp_name}}/step{final_task}/model_final.pth"
exported_final="results/ade_ps_{BACKBONE_TAG}_{suffix}_final.pth"

if [ ! -f "${{base_step_ckpt}}" ]; then
  echo "Missing shared base checkpoint: ${{base_step_ckpt}}"
  echo "Run train_base_100.sh first."
  exit 1
fi

num_prompts={prompt_count}
iter={train_iter}
base_lr=0.0005
class_weight=10.0

backbone_freeze=True
trans_decoder_freeze=True
pixel_decoder_freeze=True
cls_head_freeze=True
mask_head_freeze=True
query_embed_freeze=True
prompt_deep=True
prompt_mask_mlp=True
prompt_no_obj_mlp=False
deltas={train_deltas}

weight_args="MODEL.MASK_FORMER.NUM_OBJECT_QUERIES ${{base_queries}} MODEL.MASK_FORMER.DICE_WEIGHT ${{dice_weight}} MODEL.MASK_FORMER.MASK_WEIGHT ${{mask_weight}} MODEL.MASK_FORMER.CLASS_WEIGHT ${{class_weight}} MODEL.MASK_FORMER.SOFTMASK ${{soft_mask}} CONT.SOFTCLS ${{soft_cls}} CONT.NUM_PROMPTS ${{num_prompts}}"
comm_args="OUTPUT_DIR ${{base}} ${{meth_args}} ${{step_args}} ${{weight_args}}"
vpt_args="CONT.BACKBONE_FREEZE ${{backbone_freeze}} CONT.CLS_HEAD_FREEZE ${{cls_head_freeze}} CONT.MASK_HEAD_FREEZE ${{mask_head_freeze}} CONT.PIXEL_DECODER_FREEZE ${{pixel_decoder_freeze}} CONT.QUERY_EMBED_FREEZE ${{query_embed_freeze}} CONT.TRANS_DECODER_FREEZE ${{trans_decoder_freeze}} CONT.PROMPT_MASK_MLP ${{prompt_mask_mlp}} CONT.PROMPT_NO_OBJ_MLP ${{prompt_no_obj_mlp}} CONT.PROMPT_DEEP ${{prompt_deep}} CONT.DEEP_CLS ${{deep_cls}} CONT.LOGIT_MANI_DELTAS ${{deltas}}"

if [ -f "${{final_ckpt}}" ]; then
  echo "Final prompt checkpoint already exists, skipping training: ${{final_ckpt}}"
else
  inc_args="CONT.TASK 1 SOLVER.MAX_ITER ${{iter}} SOLVER.BASE_LR ${{base_lr}} TEST.EVAL_PERIOD {eval_period} SOLVER.CHECKPOINT_PERIOD {checkpoint_period} CONT.WEIGHTS ${{base_step_ckpt}}"
  python train_inc.py --resume --num-gpus ${{ngpus}} --config-file ${{cfg_file}} ${{comm_args}} ${{inc_args}} ${{cont_args}} ${{dist_args}} ${{vpt_args}} NAME ${{exp_name}} WANDB False
{loop}
fi

if [ ! -f "${{final_ckpt}}" ]; then
  echo "Expected final checkpoint not found: ${{final_ckpt}}"
  exit 1
fi
cp "${{final_ckpt}}" "${{exported_final}}"

deltas={eval_deltas}
vpt_args="CONT.BACKBONE_FREEZE ${{backbone_freeze}} CONT.CLS_HEAD_FREEZE ${{cls_head_freeze}} CONT.MASK_HEAD_FREEZE ${{mask_head_freeze}} CONT.PIXEL_DECODER_FREEZE ${{pixel_decoder_freeze}} CONT.QUERY_EMBED_FREEZE ${{query_embed_freeze}} CONT.TRANS_DECODER_FREEZE ${{trans_decoder_freeze}} CONT.PROMPT_MASK_MLP ${{prompt_mask_mlp}} CONT.PROMPT_NO_OBJ_MLP ${{prompt_no_obj_mlp}} CONT.PROMPT_DEEP ${{prompt_deep}} CONT.DEEP_CLS ${{deep_cls}} CONT.LOGIT_MANI_DELTAS ${{deltas}}"
inc_args="CONT.TASK {final_task} CONT.WEIGHTS ${{final_ckpt}}"
python train_inc.py --eval-only --num-gpus ${{ngpus}} --config-file ${{cfg_file}} ${{comm_args}} ${{inc_args}} ${{cont_args}} ${{dist_args}} ${{vpt_args}} NAME ${{exp_name}} WANDB False
'''

scripts = {
    "train_base_100.sh": base_script,
    "train_eval_100_50.sh": scenario_script("100_50", 50, 1, 50, PROMPT_100_50_MAX_ITER, "[0.6,-0.6]", "[-0.4,-0.6]", 4000, PROMPT_100_50_CHECKPOINT_PERIOD),
    "train_eval_100_10.sh": scenario_script("100_10", 10, 5, 10, PROMPT_100_10_MAX_ITER, "[0.4,0.5,0.5,0.5,0.5,0.5]", "[0.4,0.5,0.5,0.5,0.5,0.5]", 4000),
    "train_eval_100_5.sh": scenario_script("100_5", 5, 10, 10, PROMPT_100_5_MAX_ITER, "0.5", "[0.4,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6]", 2000),
}

for name, body in scripts.items():
    path = script_dir / name
    path.write_text(body)
    path.chmod(0o755)
    print("Wrote", path)

print("\nPreview generated base script:")
!sed -n '1,180p' "{script_dir / 'train_base_100.sh'}"


In [ ]:
# Cell 8 - Build ECLIPSE custom CUDA ops

%cd /content/ECLIPSE_CONVNEXTV2_TINY/mask2former/modeling/pixel_decoder/ops

!rm -rf build
!find . -name "*.so" -delete

# Patch old PyTorch API calls.
!sed -i 's/value.type()/value.scalar_type()/g' src/cuda/ms_deform_attn_cuda.cu
!sed -i 's/\.scalar_type()\.is_cuda()/.is_cuda()/g' src/cuda/ms_deform_attn_cuda.cu

!grep -n "AT_DISPATCH\|scalar_type\|is_cuda" src/cuda/ms_deform_attn_cuda.cu | head -30

!sh make.sh

!find . -name "*.so" -print

%cd /content/ECLIPSE_CONVNEXTV2_TINY

%env PYTHONPATH=/content/detectron2_src:/content/ECLIPSE_CONVNEXTV2_TINY:/content/ECLIPSE_CONVNEXTV2_TINY/mask2former/modeling/pixel_decoder/ops

import sys
import torch

print("Python:", sys.executable)
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)

# Make notebook kernel see Detectron2 source
if "/content/detectron2_src" not in sys.path:
    sys.path.insert(0, "/content/detectron2_src")

if "/content/ECLIPSE_CONVNEXTV2_TINY/mask2former/modeling/pixel_decoder/ops" not in sys.path:
    sys.path.insert(0, "/content/ECLIPSE_CONVNEXTV2_TINY/mask2former/modeling/pixel_decoder/ops")

from detectron2 import _C
print("Detectron2 _C OK")

import MultiScaleDeformableAttention
print("ECLIPSE MultiScaleDeformableAttention OK")


In [ ]:
# Cell 9 - Sanity check before training

%cd /content/ECLIPSE_CONVNEXTV2_TINY

from pathlib import Path
import sys
import os
import torch

try:
    BACKBONE_TAG
    CONVNEXTV2_TIMM_MODEL
except NameError:
    BACKBONE_TAG = "convnextv2_t"
    CONVNEXTV2_TIMM_MODEL = "convnextv2_tiny.fcmae_ft_in22k_in1k"

required = [
    Path("train_inc.py"),
    Path("datasets/ADEChallengeData2016"),
    Path("mask2former/modeling/backbone/convnextv2.py"),
    Path(f"configs/ade20k/panoptic-segmentation/convnextv2/maskformer2_{BACKBONE_TAG}_bs16_160k.yaml"),
    Path(f"script/ade_ps_{BACKBONE_TAG}/train_base_100.sh"),
    Path(f"script/ade_ps_{BACKBONE_TAG}/train_eval_100_50.sh"),
    Path(f"script/ade_ps_{BACKBONE_TAG}/train_eval_100_10.sh"),
    Path(f"script/ade_ps_{BACKBONE_TAG}/train_eval_100_5.sh"),
]

missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing files/folders:\n" + "\n".join(missing))

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ops_root = Path("/content/ECLIPSE_CONVNEXTV2_TINY/mask2former/modeling/pixel_decoder/ops")
so_files = list(ops_root.rglob("MultiScaleDeformableAttention*.so"))
print("\nFound ECLIPSE custom op files:")
for so in so_files:
    print(so.resolve())
if not so_files:
    raise FileNotFoundError("MultiScaleDeformableAttention .so was not found. Rerun Cell 8.")

so_dir = str(so_files[0].resolve().parent)
paths_to_add = ["/content/detectron2_src", "/content/ECLIPSE_CONVNEXTV2_TINY", so_dir]
for path in paths_to_add:
    if path not in sys.path:
        sys.path.insert(0, path)
os.environ["PYTHONPATH"] = ":".join(paths_to_add + [os.environ.get("PYTHONPATH", "")])
os.environ["CONVNEXTV2_TIMM_MODEL"] = CONVNEXTV2_TIMM_MODEL

import detectron2
from detectron2 import _C
print("\nDetectron2 _C OK")

import MultiScaleDeformableAttention
print("ECLIPSE MultiScaleDeformableAttention OK")

from mask2former.modeling.backbone.convnextv2 import D2ConvNeXtV2Tiny
print("ConvNeXt-V2 backbone registration import OK")
print("\nNo-TTA ConvNeXt-V2-Tiny training setup is ready.")


In [ ]:
# Cell 10 - RESTORE OR CONTINUE CHECKPOINTS before training

%cd /content/ECLIPSE_CONVNEXTV2_TINY

from pathlib import Path
import shutil
import subprocess

# RUN THIS CELL RIGHT BEFORE TRAINING.
# It restores saved checkpoints from Google Drive into the exact folders that
# Detectron2/ECLIPSE checks when --resume is used. This is what lets you continue
# base or 100-50 training after Colab disconnects.
#
# What gets restored/saved:
# - Final exported checkpoints for base, 100-50, 100-10, and 100-5.
# - Live 10k checkpoints for base training, including last_checkpoint.
# - Live 10k checkpoints for 100-50 training, including last_checkpoint.
#
# While Cell 11 and Cell 12 are training, background sync loops copy new 10k
# checkpoints to Drive every few minutes. If Colab dies, rerun cells 0-10, then
# run the training cell again; the generated scripts use --resume and continue.

try:
    BACKBONE_TAG
except NameError:
    BACKBONE_TAG = "convnextv2_t"

backbone_tag = BACKBONE_TAG
CHECKPOINT_BACKUP_DIR = Path("/content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_CHECKPOINTS") / backbone_tag
CHECKPOINT_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

RESULT_ROOT = Path(f"results/ade_ps_{backbone_tag}")
BASE_OUTPUT_DIR = RESULT_ROOT / "mya-pan_100-50-ov" / f"adps_{backbone_tag}_base100" / "step0"
PROMPT_100_50_OUTPUT_DIR = RESULT_ROOT / "mya-pan_100-50-ov" / f"adps_{backbone_tag}_100_50" / "step1"

LIVE_STAGE_DIRS = {
    "base100": {"local": BASE_OUTPUT_DIR, "backup": CHECKPOINT_BACKUP_DIR / "base100_live_10k", "label": "base training"},
    "100_50": {"local": PROMPT_100_50_OUTPUT_DIR, "backup": CHECKPOINT_BACKUP_DIR / "100_50_live_10k", "label": "100-50 training"},
}
for stage in LIVE_STAGE_DIRS.values():
    stage["backup"].mkdir(parents=True, exist_ok=True)

CHECKPOINT_MANIFEST = {
    "base100": [Path(f"results/ade_ps_{backbone_tag}_100_step0.pth"), BASE_OUTPUT_DIR / "model_final.pth"],
    "100_50_final": [Path(f"results/ade_ps_{backbone_tag}_100_50_final.pth"), PROMPT_100_50_OUTPUT_DIR / "model_final.pth"],
    "100_10_final": [Path(f"results/ade_ps_{backbone_tag}_100_10_final.pth"), RESULT_ROOT / "mya-pan_100-10-ov" / f"adps_{backbone_tag}_100_10" / "step5" / "model_final.pth"],
    "100_5_final": [Path(f"results/ade_ps_{backbone_tag}_100_5_final.pth"), RESULT_ROOT / "mya-pan_100-5-ov" / f"adps_{backbone_tag}_100_5" / "step10" / "model_final.pth"],
}


def restore_convnextv2_final_checkpoints():
    # Restore final stage checkpoints. These make finished stages skip retraining.
    restored = []
    for name, local_paths in CHECKPOINT_MANIFEST.items():
        backup = CHECKPOINT_BACKUP_DIR / f"{name}.pth"
        if not backup.exists():
            continue
        for local_path in local_paths:
            local_path.parent.mkdir(parents=True, exist_ok=True)
            if not local_path.exists():
                shutil.copy2(backup, local_path)
                restored.append(str(local_path))
    if restored:
        print("Restored final checkpoints from Drive:")
        for item in restored:
            print("  ", item)
    else:
        print("No final checkpoints needed restoring.")


def restore_convnextv2_live_checkpoints(stage_name=None):
    # Restore intermediate model_*.pth files plus last_checkpoint. These let
    # --resume continue unfinished base or 100-50 training.
    names = [stage_name] if stage_name else list(LIVE_STAGE_DIRS)
    for name in names:
        stage = LIVE_STAGE_DIRS[name]
        local_dir = stage["local"]
        backup_dir = stage["backup"]
        local_dir.mkdir(parents=True, exist_ok=True)
        restored = []
        for source in backup_dir.glob("*"):
            if source.is_file() and (source.name.endswith(".pth") or source.name == "last_checkpoint"):
                target = local_dir / source.name
                if not target.exists() or source.stat().st_mtime > target.stat().st_mtime or source.stat().st_size != target.stat().st_size:
                    shutil.copy2(source, target)
                    restored.append(str(target))
        if restored:
            print(f"Restored live checkpoints for {stage['label']}:")
            for item in restored:
                print("  ", item)
        else:
            print(f"No live checkpoints needed restoring for {stage['label']}.")


def sync_convnextv2_live_checkpoints_once(stage_name):
    # Copy current local intermediate checkpoints for one stage to Drive.
    stage = LIVE_STAGE_DIRS[stage_name]
    local_dir = stage["local"]
    backup_dir = stage["backup"]
    if not local_dir.exists():
        print(f"{stage['label']} output folder does not exist yet; nothing to sync:", local_dir)
        return
    backup_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    for source in local_dir.glob("*"):
        if not source.is_file():
            continue
        if not ((source.name.startswith("model_") and source.suffix == ".pth") or source.name == "last_checkpoint"):
            continue
        target = backup_dir / source.name
        if not target.exists() or source.stat().st_size != target.stat().st_size or source.stat().st_mtime > target.stat().st_mtime:
            shutil.copy2(source, target)
            copied.append((str(source), str(target)))
    if copied:
        print(f"Synced live checkpoints for {stage['label']} to Drive:")
        for source, target in copied:
            print(f"  {source} -> {target}")
    else:
        print(f"No new live checkpoints needed syncing for {stage['label']}.")


_convnextv2_live_sync_processes = {}


def start_convnextv2_live_checkpoint_sync(stage_name, interval_seconds=180):
    # Start one background Drive sync loop for base or 100-50 before training.
    stage = LIVE_STAGE_DIRS[stage_name]
    existing = _convnextv2_live_sync_processes.get(stage_name)
    if existing is not None and existing.poll() is None:
        print(f"Live sync is already running for {stage['label']}.")
        return existing

    local_dir = stage["local"]
    backup_dir = stage["backup"]
    local_dir.mkdir(parents=True, exist_ok=True)
    backup_dir.mkdir(parents=True, exist_ok=True)
    script = f'''
set -u
src="{local_dir}"
dst="{backup_dir}"
mkdir -p "$dst"
echo "[convnextv2-sync] Watching $src"
echo "[convnextv2-sync] Copying checkpoints to $dst every {interval_seconds}s"
while true; do
  if [ -d "$src" ]; then
    find "$src" -maxdepth 1 -type f \\( -name 'model_*.pth' -o -name 'model_final.pth' -o -name 'last_checkpoint' \\) -print0 | while IFS= read -r -d '' f; do
      cp -u "$f" "$dst/"
    done
  fi
  sleep {int(interval_seconds)}
done
'''
    process = subprocess.Popen(["bash", "-lc", script])
    _convnextv2_live_sync_processes[stage_name] = process
    print(f"Started live sync for {stage['label']}, pid: {process.pid}")
    return process


def stop_convnextv2_live_checkpoint_sync(stage_name):
    # Stop the background sync loop and do one final copy pass.
    process = _convnextv2_live_sync_processes.get(stage_name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
        print(f"Stopped live sync for {LIVE_STAGE_DIRS[stage_name]['label']}.")
    _convnextv2_live_sync_processes[stage_name] = None
    sync_convnextv2_live_checkpoints_once(stage_name)


def save_convnextv2_training_checkpoints():
    # Save live checkpoints and final exported checkpoints after each training cell.
    for stage_name in LIVE_STAGE_DIRS:
        sync_convnextv2_live_checkpoints_once(stage_name)
    saved = []
    for name, local_paths in CHECKPOINT_MANIFEST.items():
        source = next((path for path in local_paths if path.exists()), None)
        if source is None:
            continue
        backup = CHECKPOINT_BACKUP_DIR / f"{name}.pth"
        if (not backup.exists()) or source.stat().st_size != backup.stat().st_size or source.stat().st_mtime > backup.stat().st_mtime:
            shutil.copy2(source, backup)
            saved.append((str(source), str(backup)))
    if saved:
        print("Saved final checkpoints to Drive:")
        for source, backup in saved:
            print(f"  {source} -> {backup}")
    else:
        print("No new final checkpoints needed saving.")


restore_convnextv2_final_checkpoints()
restore_convnextv2_live_checkpoints()
print("Final checkpoint backup folder:", CHECKPOINT_BACKUP_DIR)
for name, stage in LIVE_STAGE_DIRS.items():
    print(f"Live checkpoint backup folder for {stage['label']}:", stage["backup"])
print("Now run Cell 11 for base or Cell 12 for 100-50. Both will continue from restored checkpoints if available.")


In [ ]:
# Cell 11 - Train/resume the shared 100-class ConvNeXt-V2-Tiny base step

%cd /content/ECLIPSE_CONVNEXTV2_TINY

from pathlib import Path
import os

backbone_tag = BACKBONE_TAG
ops_root = Path("/content/ECLIPSE_CONVNEXTV2_TINY/mask2former/modeling/pixel_decoder/ops")
so_files = list(ops_root.rglob("MultiScaleDeformableAttention*.so"))
if not so_files:
    raise FileNotFoundError("MultiScaleDeformableAttention .so was not found. Rerun Cell 8.")
so_dir = str(so_files[0].resolve().parent)
pythonpath = f"/content/detectron2_src:/content/ECLIPSE_CONVNEXTV2_TINY:{so_dir}:{os.environ.get('PYTHONPATH', '')}"

!mkdir -p /content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_LOGS


# Start live Drive sync before the blocking base-training command.
# While train_base_100.sh is running, this background process copies each new
# 10k base checkpoint from the ECLIPSE output folder to Google Drive.
try:
    start_convnextv2_live_checkpoint_sync("base100", interval_seconds=180)
except NameError:
    print("Run Cell 10 to restore checkpoints and enable live ConvNeXt-V2 base checkpoint syncing.")

!CONVNEXTV2_TIMM_MODEL="{CONVNEXTV2_TIMM_MODEL}" PYTHONPATH="{pythonpath}" bash "script/ade_ps_{backbone_tag}/train_base_100.sh" 2>&1 | tee "output_convnextv2_t_base100.txt"
!cp "output_convnextv2_t_base100.txt" /content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_LOGS/


# Stop the live sync after training/evaluation finishes, then do one final copy pass.
try:
    stop_convnextv2_live_checkpoint_sync("base100")
except NameError:
    pass



# Cell 10 defines save_convnextv2_training_checkpoints().
# We call it after the stage finishes so the final .pth is copied to Drive.
# On a later runtime, Cell 10 restores that .pth into the exact path that the
# bash script checks, so the script skips retraining and only evaluates if needed.
try:
    save_convnextv2_training_checkpoints()
except NameError:
    print("Run Cell 10 to enable ConvNeXt-V2 checkpoint saving/restoring.")


print("Finished or reused shared ConvNeXt-V2-Tiny base checkpoint.")


In [ ]:
# Cell 12 - Train/resume and evaluate ConvNeXt-V2-Tiny 100-50, no TTA

%cd /content/ECLIPSE_CONVNEXTV2_TINY

from pathlib import Path
import os

backbone_tag = BACKBONE_TAG
ops_root = Path("/content/ECLIPSE_CONVNEXTV2_TINY/mask2former/modeling/pixel_decoder/ops")
so_files = list(ops_root.rglob("MultiScaleDeformableAttention*.so"))
if not so_files:
    raise FileNotFoundError("MultiScaleDeformableAttention .so was not found. Rerun Cell 8.")
so_dir = str(so_files[0].resolve().parent)
pythonpath = f"/content/detectron2_src:/content/ECLIPSE_CONVNEXTV2_TINY:{so_dir}:{os.environ.get('PYTHONPATH', '')}"

!mkdir -p /content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_LOGS


# Start live Drive sync before the blocking 100-50 training command.
# Cell 7 generates this script with SOLVER.CHECKPOINT_PERIOD 10000, so this
# background process copies each new 10k 100-50 checkpoint to Google Drive.
try:
    start_convnextv2_live_checkpoint_sync("100_50", interval_seconds=180)
except NameError:
    print("Run Cell 10 to restore checkpoints and enable live ConvNeXt-V2 100-50 checkpoint syncing.")

!CONVNEXTV2_TIMM_MODEL="{CONVNEXTV2_TIMM_MODEL}" PYTHONPATH="{pythonpath}" bash "script/ade_ps_{backbone_tag}/train_eval_100_50.sh" 2>&1 | tee "output_convnextv2_t_100_50.txt"
!cp "output_convnextv2_t_100_50.txt" /content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_LOGS/


# Stop the 100-50 live sync after training/evaluation finishes, then do one final copy pass.
try:
    stop_convnextv2_live_checkpoint_sync("100_50")
except NameError:
    pass



# Cell 10 defines save_convnextv2_training_checkpoints().
# We call it after the stage finishes so the final .pth is copied to Drive.
# On a later runtime, Cell 10 restores that .pth into the exact path that the
# bash script checks, so the script skips retraining and only evaluates if needed.
try:
    save_convnextv2_training_checkpoints()
except NameError:
    print("Run Cell 10 to enable ConvNeXt-V2 checkpoint saving/restoring.")


print("Finished ConvNeXt-V2-Tiny 100-50 training/evaluation.")


In [ ]:
# Cell 13 - Train and evaluate ConvNeXt-V2-Tiny 100-10, no TTA

%cd /content/ECLIPSE_CONVNEXTV2_TINY

from pathlib import Path
import os

backbone_tag = BACKBONE_TAG
ops_root = Path("/content/ECLIPSE_CONVNEXTV2_TINY/mask2former/modeling/pixel_decoder/ops")
so_files = list(ops_root.rglob("MultiScaleDeformableAttention*.so"))
if not so_files:
    raise FileNotFoundError("MultiScaleDeformableAttention .so was not found. Rerun Cell 8.")
so_dir = str(so_files[0].resolve().parent)
pythonpath = f"/content/detectron2_src:/content/ECLIPSE_CONVNEXTV2_TINY:{so_dir}:{os.environ.get('PYTHONPATH', '')}"

!mkdir -p /content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_LOGS
!CONVNEXTV2_TIMM_MODEL="{CONVNEXTV2_TIMM_MODEL}" PYTHONPATH="{pythonpath}" bash "script/ade_ps_{backbone_tag}/train_eval_100_10.sh" 2>&1 | tee "output_convnextv2_t_100_10.txt"
!cp "output_convnextv2_t_100_10.txt" /content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_LOGS/


# Cell 10 defines save_convnextv2_training_checkpoints().
# We call it after the stage finishes so the final .pth is copied to Drive.
# On a later runtime, Cell 10 restores that .pth into the exact path that the
# bash script checks, so the script skips retraining and only evaluates if needed.
try:
    save_convnextv2_training_checkpoints()
except NameError:
    print("Run Cell 10 to enable ConvNeXt-V2 checkpoint saving/restoring.")


print("Finished ConvNeXt-V2-Tiny 100-10 training/evaluation.")


In [ ]:
# Cell 14 - Train and evaluate ConvNeXt-V2-Tiny 100-5, no TTA

%cd /content/ECLIPSE_CONVNEXTV2_TINY

from pathlib import Path
import os

backbone_tag = BACKBONE_TAG
ops_root = Path("/content/ECLIPSE_CONVNEXTV2_TINY/mask2former/modeling/pixel_decoder/ops")
so_files = list(ops_root.rglob("MultiScaleDeformableAttention*.so"))
if not so_files:
    raise FileNotFoundError("MultiScaleDeformableAttention .so was not found. Rerun Cell 8.")
so_dir = str(so_files[0].resolve().parent)
pythonpath = f"/content/detectron2_src:/content/ECLIPSE_CONVNEXTV2_TINY:{so_dir}:{os.environ.get('PYTHONPATH', '')}"

!mkdir -p /content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_LOGS
!CONVNEXTV2_TIMM_MODEL="{CONVNEXTV2_TIMM_MODEL}" PYTHONPATH="{pythonpath}" bash "script/ade_ps_{backbone_tag}/train_eval_100_5.sh" 2>&1 | tee "output_convnextv2_t_100_5.txt"
!cp "output_convnextv2_t_100_5.txt" /content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_LOGS/


# Cell 10 defines save_convnextv2_training_checkpoints().
# We call it after the stage finishes so the final .pth is copied to Drive.
# On a later runtime, Cell 10 restores that .pth into the exact path that the
# bash script checks, so the script skips retraining and only evaluates if needed.
try:
    save_convnextv2_training_checkpoints()
except NameError:
    print("Run Cell 10 to enable ConvNeXt-V2 checkpoint saving/restoring.")


print("Finished ConvNeXt-V2-Tiny 100-5 training/evaluation.")


In [ ]:
# Cell 15 - Extract final all-PQ lines for all ConvNeXt-V2-Tiny scenarios

%cd /content/ECLIPSE_CONVNEXTV2_TINY

print("===== 100-50 =====")
!grep -A 3 -i "copypaste: PQ" "output_convnextv2_t_100_50.txt" || true

print("\n===== 100-10 =====")
!grep -A 3 -i "copypaste: PQ" "output_convnextv2_t_100_10.txt" || true

print("\n===== 100-5 =====")
!grep -A 3 -i "copypaste: PQ" "output_convnextv2_t_100_5.txt" || true


In [ ]:
# Cell 16 - Graph article results vs final ConvNeXt-V2-Tiny results

%cd /content/ECLIPSE_CONVNEXTV2_TINY

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import re

backbone_label = "ConvNeXt-V2-Tiny"

scenarios = ['100-5\n(11 Tasks)', '100-10\n(6 Tasks)', '100-50\n(2 Tasks)']
article_scores = [32.9, 33.9, 35.6]
log_files = [
    Path('output_convnextv2_t_100_5.txt'),
    Path('output_convnextv2_t_100_10.txt'),
    Path('output_convnextv2_t_100_50.txt'),
]

def parse_all_pq(log_path):
    if not log_path.exists():
        raise FileNotFoundError(f"Missing {log_path}. Run the corresponding train/eval cell first.")
    lines = log_path.read_text(errors='ignore').splitlines()
    for idx, line in enumerate(lines):
        if 'copypaste:' in line.lower() and re.search(r'\bPQ\b', line):
            for candidate in lines[idx + 1:idx + 6]:
                if 'copypaste:' not in candidate.lower():
                    continue
                numbers = re.findall(r'[-+]?\d+(?:\.\d+)?', candidate)
                if numbers:
                    return float(numbers[0])
    raise ValueError(f"Could not parse all-PQ from {log_path}")

convnext_scores = [parse_all_pq(p) for p in log_files]

x = np.arange(len(scenarios))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 6))
r1 = ax.bar(x - width / 2, article_scores, width, label='Original article')
r2 = ax.bar(x + width / 2, convnext_scores, width, label=f'{backbone_label}, no TTA')

ax.set_ylabel('All PQ')
ax.set_title('ECLIPSE ADE20K-Panoptic: Original vs ConvNeXt-V2-Tiny')
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.set_ylim(30, max(article_scores + convnext_scores) + 2)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.45)
ax.bar_label(r1, padding=3, fmt='%.2f')
ax.bar_label(r2, padding=3, fmt='%.2f')

plt.tight_layout()
out_dir = Path('/content/drive/MyDrive/ECLIPSE_CONVNEXTV2_TINY_LOGS')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'eclipse_convnextv2_t_no_tta_article_comparison.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()

print('Original article all-PQ:', article_scores)
print('ConvNeXt-V2-Tiny no-TTA all-PQ:', convnext_scores)
print('Saved graph to:', out_path)


## Report wording

Use this after the notebook finishes:

> We reproduced the ADE20K-Panoptic continual segmentation setup from ECLIPSE and changed only the frozen-base-model backbone path. Instead of the original ResNet-50 backbone, we trained a new 100-class base step with a ConvNeXt-V2-Tiny backbone initialized from timm pretrained weights, then froze the base model and trained the continual prompt steps for 100-50, 100-10, and 100-5. Evaluation used the normal single-scale inference path with no test-time augmentation. The cleaned notebook restores checkpoints before training and live-syncs 10k-interval checkpoints for both base and 100-50 training to Google Drive, so interrupted Colab runs can resume from the latest saved checkpoint. We compare the final all-PQ values from the new logs against the original article values.

Important: the graph parses the final **all PQ** value from each fresh ConvNeXt-V2-Tiny evaluation log. If you lower iteration counts for a smoke test, do not report those graph values as final results.
